# ofiq-syngen Cookbook

Self-contained walkthrough of every public surface in `ofiq-syngen`. Runs end-to-end on a synthetic face image so it has no external dependencies beyond the package itself.

Sections:
1. Setup
2. Single degradation
3. Severity sweep
4. All 28 components at once
5. Inspecting the registry
6. Standards cross-reference
7. Standards-preset filtering
8. Per-component standards lookup
9. Conformance bundle

For real face images you would replace the synthetic fixture in section 1 with `cv2.imread('your_face.jpg')`.

## 1. Setup

In [ ]:
import numpy as np
import cv2
from ofiq_syngen import DegradationPipeline, DegradationConfig
from ofiq_syngen.components import COMPONENT_REGISTRY, list_supported_components
from ofiq_syngen.standards import (
    STANDARDS_REFS, ICAO_STRICT_COMPONENTS,
    get_refs, components_by_alignment, components_by_confidence,
    components_for_ofiq_version,
)

# Synthetic face: 256x256 BGR uint8
rng = np.random.RandomState(0)
face = rng.randint(80, 200, (256, 256, 3), dtype=np.uint8)
print(f'face shape: {face.shape}, dtype: {face.dtype}')
print(f'{len(list_supported_components())} components registered')

## 2. Single degradation

In [ ]:
pipeline = DegradationPipeline()
degraded, meta = pipeline.degrade_single(face, 'Sharpness.scalar', 0.6, seed=42)
print('meta:', meta)
print('degraded shape:', degraded.shape, 'mean delta:', float(np.abs(degraded.astype(int) - face.astype(int)).mean()))

## 3. Severity sweep

In [ ]:
config = DegradationConfig(severity_levels=[0.0, 0.25, 0.5, 0.75, 1.0])
p = DegradationPipeline(config)
results = p.degrade_sweep(face, 'CompressionArtifacts.scalar', seed=42)
for img, m in results:
    delta = float(np.abs(img.astype(int) - face.astype(int)).mean())
    print(f"  severity {m['severity']:.2f}  mean delta {delta:6.2f}")

## 4. All 28 components at once

In [ ]:
all_results = pipeline.degrade_all_components(face, severity=0.5, seed=42)
print(f'returned {len(all_results)} (image, meta) tuples')
components_seen = sorted({m['target_component'] for _, m in all_results})
print(f'distinct components: {len(components_seen)}')
print('first three:', components_seen[:3])

## 5. Inspecting the registry

In [ ]:
for comp, degs in sorted(COMPONENT_REGISTRY.items())[:5]:
    for d in degs:
        flag = ' (ctx)' if d.requires_context else ''
        print(f'{comp:40s}{flag:6s}  {d.description}')

## 6. Standards cross-reference

In [ ]:
refs = get_refs('Sharpness.scalar')
print(f'OFIQ section:  {refs.ofiq_section}')
print(f'ISO 29794-5:   {refs.iso_29794_5}')
print(f'ISO 19794-5:   {refs.iso_19794_5}')
print(f'ICAO 9303 P9:  {refs.icao_9303}')
print(f'alignment:     {refs.alignment}')
print(f'confidence:    {refs.confidence}')
print(f'OFIQ versions: {refs.ofiq_versions}')

## 7. Standards-preset filtering

In [ ]:
print(f'icao-strict preset: {len(ICAO_STRICT_COMPONENTS)} components')
for c in ICAO_STRICT_COMPONENTS[:5]:
    print(f'  {c}')
print(f'\nalignment=partial: {components_by_alignment("partial")}')
print(f'confidence=uncertain: {components_by_confidence("uncertain")}')
print(f'\ncomponents valid for OFIQ 1.1: {len(components_for_ofiq_version("1.1"))}')

## 8. Per-component standards lookup

In [ ]:
for comp, refs in sorted(STANDARDS_REFS.items())[:10]:
    print(f'{comp:40s}  {refs.ofiq_section:5s}  {refs.alignment:7s}  {refs.confidence}')

## 9. Conformance bundle

Export a ZIP that any OFIQ-aligned implementation can use to validate its own measurement chain. Contains the standards mapping, OFIQ source provenance, and the parity manifest (empty by default; see `tests/fixtures/ofiq_parity/REGENERATE.md` to populate).

In [ ]:
import subprocess, sys
rc = subprocess.call([sys.executable, '-m', 'ofiq_syngen.cli', 'export-conformance', '-o', '/tmp/conformance.zip'])
print('exit:', rc)
import zipfile
with zipfile.ZipFile('/tmp/conformance.zip') as zf:
    for info in zf.infolist():
        print(f'  {info.filename:30s}  {info.file_size} bytes')

## Next steps

- Replace the synthetic face in section 1 with a real face image.
- Install OFIQ ONNX models (`pip install ofiq-syngen[gpu]` plus the model bundle from BSI) to enable `FaceContext`-aware degradation.
- Run `examples/realtime_capture.py` for a webcam demo.
- See `docs/standards/MAPPING.md` for the full multi-standard cross-reference.